# RL Video Summarization — End-to-End Pipeline Test

Run from the **project root** (`RL-Video-Summarization/`) so relative paths resolve correctly.

```bash
cd RL-Video-Summarization
jupyter notebook test_pipeline_e2e.ipynb
```

**Test order:**
1. Imports & sys.path
2. Data loader
3. Transformer encoder
4. Feature pipeline
5. MDP components (State, Reward, Environment)
6. Policies (Horizontal, Vertical)
7. REINFORCE trainer (single video)
8. End-to-end multi-video training loop
9. Checkpoint save / load

## Setup Instructions

```bash
cd RL-Video-Summarization
jupyter notebook test_pipeline_e2e.ipynb
```


## 0. Setup — sys.path & seeds

In [1]:
import sys, os
import numpy as np
import torch

PROJECT_ROOT = os.getcwd()  # must be RL-Video-Summarization/
MDP_PATH     = os.path.join(PROJECT_ROOT, 'MDP')

for p in [PROJECT_ROOT, MDP_PATH]:
    if p not in sys.path:
        sys.path.insert(0, p)

print('PROJECT_ROOT:', PROJECT_ROOT)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__} | device: {DEVICE}')

PROJECT_ROOT: /home/jovyan/RL-Video-Summarization
PyTorch 2.11.0+cu130 | device: cuda


In [2]:
# === DIAGNOSTIC: Check if data is accessible ===
import os
from pathlib import Path

print("=" * 60)
print("DATA LOADER DIAGNOSTIC")
print("=" * 60)
print(f"Current working directory: {os.getcwd()}")
print(f"\nChecking for video embeddings...")

candidates = [
    Path.cwd() / 'videos' / 'embeddings_clip_vitl14',
    Path.home() / 'videos' / 'embeddings_clip_vitl14',
    Path('/scratch/videos/embeddings_clip_vitl14'),
    Path('/data/videos/embeddings_clip_vitl14'),
]

found = False
for path in candidates:
    exists = path.exists()
    status = '✓' if exists else '✗'
    print(f"  {status} {path}")
    if exists:
        found = True
        print(f"    → Using this path!")

if 'VIDEOS_DIR' in os.environ:
    env_path = Path(os.environ['VIDEOS_DIR'])
    status = '✓' if env_path.exists() else '✗'
    print(f"  {status} VIDEOS_DIR={env_path}")
    found = found or env_path.exists()

if not found:
    print("\nWARNING: No video embeddings found!")
    print("On HPC, set the environment variable before running this notebook:")
    print("  export VIDEOS_DIR=/path/to/embeddings_clip_vitl14")
    print("  jupyter notebook test_pipeline_e2e.ipynb")
else:
    print("\nVideos found! Continue with the notebook.")
print("=" * 60)


DATA LOADER DIAGNOSTIC
Current working directory: /home/jovyan/RL-Video-Summarization

Checking for video embeddings...
  ✓ /home/jovyan/RL-Video-Summarization/videos/embeddings_clip_vitl14
    → Using this path!
  ✗ /home/jovyan/videos/embeddings_clip_vitl14
  ✗ /scratch/videos/embeddings_clip_vitl14
  ✗ /data/videos/embeddings_clip_vitl14

✅ Videos found! Continue with the notebook.


## 1. Config

In [3]:
from training.config import Config

cfg = Config()
print(cfg)

assert cfg.transformer_d_model % cfg.transformer_nhead == 0
assert cfg.d_model == cfg.transformer_d_model
print('\nConfig OK')

Config(transformer_input_dim=768, transformer_d_model=512, transformer_nhead=8, transformer_num_layers=3, transformer_dim_feedfwd=2048, transformer_dropout=0.1, transformer_max_len=5000, d_model=512, hidden_size=256, alpha=0.02, min_k=3, max_k=20, window_size=5, max_steps=100, stability_patience=5, w_div=0.5, w_rep=0.5, lr=0.001, gamma=1.0, seed=42)

Config OK


## 2. Data Loader

In [4]:
from training.data_loader import VideoEmbeddingLoader

loader = VideoEmbeddingLoader(videos_dir='videos/embeddings_clip_vitl14')
categories = loader.get_categories()
print(f'Categories ({len(categories)}): {categories}')

[VideoEmbeddingLoader] Found 3 categories
[VideoEmbeddingLoader] Categories: Activity, Animals, Incidents
[VideoEmbeddingLoader] Videos directory: /home/jovyan/RL-Video-Summarization/videos/embeddings_clip_vitl14
Categories (3): ['Activity', 'Animals', 'Incidents']


In [5]:
print(f'{"Category":<15} {"Videos":>7} {"Min T":>7} {"Max T":>7} {"Avg T":>7}')

for cat in categories:
    s = loader.get_statistics(cat)
    print(f'{cat:<15} {s["num_videos"]:>7} {s["min_length"]:>7} {s["max_length"]:>7} {s["avg_length"]:>7.1f}')

Category         Videos   Min T   Max T   Avg T
Activity             77      22     466   253.4
Animals               9      25     158    88.0
Incidents            43      23     668   266.2


In [6]:
test_cat = categories[0]
videos   = loader.list_videos_in_category(test_cat)
emb      = loader.load_single_video(f'{test_cat}/{videos[0]}')

print(f'Shape: {emb.shape}  # expected (T, 768)')
assert emb.ndim == 2
assert emb.shape[1] == cfg.transformer_input_dim
print('Data loader OK')

Shape: (370, 768)  # expected (T, 768)
Data loader OK


## 3. Transformer Encoder

In [7]:
from transformer.transformer_encoder import TemporalTransformerEncoder, pad_and_mask

B, T, D_IN, D_OUT = 2, 50, 768, 512
enc = TemporalTransformerEncoder(input_dim=D_IN, d_model=D_OUT).to(DEVICE)

x   = torch.randn(B, T, D_IN).to(DEVICE)
out = enc.encode(x)
print(f'Input:  {x.shape}  →  Output: {out.shape}  (expected ({B},{T},{D_OUT}))')
assert out.shape == (B, T, D_OUT)
print('Encoder forward OK')

/home/jovyan/RL-Video-Summarization/transformer/transformer_encoder.py:143: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Input:  torch.Size([2, 50, 768])  →  Output: torch.Size([2, 50, 512])  (expected (2,50,512))
Encoder forward OK


In [8]:
fl = [torch.randn(30, D_IN), torch.randn(50, D_IN), torch.randn(20, D_IN)]
padded, mask = pad_and_mask(fl, d_input=D_IN)
print(f'Padded: {padded.shape}  Mask: {mask.shape}')
print(f'Real frames: {(~mask).sum(dim=1).tolist()}  (expected [30, 50, 20])')
assert (~mask).sum(dim=1).tolist() == [30, 50, 20]
print('pad_and_mask OK')

Padded: torch.Size([3, 50, 768])  Mask: torch.Size([3, 50])
Real frames: [30, 50, 20]  (expected [30, 50, 20])
pad_and_mask OK


## 4. Feature Pipeline

In [9]:
from training.feature_pipeline import FeaturePipeline

pipeline = FeaturePipeline(cfg, device=DEVICE)

raw = np.random.randn(120, cfg.transformer_input_dim).astype(np.float32)
ctx = pipeline.encode_video(raw)
print(f'Raw: {raw.shape}  →  Ctx: {ctx.shape}  (expected (120, 512))')
assert ctx.shape == (120, cfg.transformer_d_model)
print('Single-video encode OK')

[FeaturePipeline] Transformer initialized on cuda
  Temporal Transformer Encoder
  Input dimension    : 768
  Output dimension   : 512
  Attention heads    : 8
  Encoder layers     : 3
  FFN hidden size    : 2048
  Total parameters   : 9,852,928
  Device             : cuda
Raw: (120, 768)  →  Ctx: (120, 512)  (expected (120, 512))
Single-video encode OK


In [10]:
raw_list = [
    np.random.randn(80,  cfg.transformer_input_dim).astype(np.float32),
    np.random.randn(120, cfg.transformer_input_dim).astype(np.float32),
    np.random.randn(60,  cfg.transformer_input_dim).astype(np.float32),
]
padded_ctx, batch_mask = pipeline.encode_batch(raw_list)
print(f'Batch ctx: {padded_ctx.shape}  Mask: {batch_mask.shape}')
print(f'Real frames: {(~batch_mask).sum(axis=1).tolist()}  (expected [80, 120, 60])')
assert padded_ctx.shape == (3, 120, cfg.transformer_d_model)
print('Batch encode OK')

Batch ctx: (3, 120, 512)  Mask: (3, 120)
Real frames: [80, 120, 60]  (expected [80, 120, 60])
Batch encode OK


## 5. MDP Components

In [11]:
from state import State

s = State([0, 5, 10, 15], anchor_idx=5)
print('Initial:', s)

assert s.replace_anchor(20) == True
assert 5 not in s.selected_indices and 20 in s.selected_indices
assert s.replace_anchor(20) == False  # no-op

s2 = s.copy()
s2.add_frame(99)
assert 99 not in s.selected_indices
print('State OK')

Initial: State(frame_indices=[0, 10, 5, 15], anchor=5)
State OK


In [12]:
from reward import Reward

rwd = Reward(0.5, 0.5)
F   = np.random.randn(50, cfg.d_model).astype(np.float32)
sel = {0, 10, 20, 30}

total = rwd.compute_total_reward(F, sel)
print(f'Total reward: {total:.4f}  (expected in [0,1])')
assert 0.0 <= total <= 1.0
assert rwd.compute_total_reward(F, set()) == 0.0
assert rwd.compute_diversity_reward(F[:1]) == 0.0
print('Reward OK')

Total reward: 0.5655  (expected in [0,1])
Reward OK


In [13]:
from environment import VideoSummarizationEnv

ctx = np.random.randn(100, cfg.d_model).astype(np.float32)
env = VideoSummarizationEnv(ctx, cfg)
obs = env.reset(seed=SEED)

assert obs['turn'] == 'H'
k_exp = max(cfg.min_k, min(int(100 * cfg.alpha), cfg.max_k))
assert len(obs['summary_indices']) == k_exp

# H step
anchor = obs['summary_indices'][0]
obs2, r_h, done_h, _ = env.step_H(anchor)
assert r_h == 0.0 and not done_h and obs2['turn'] == 'V'

# V step
obs3, r_v, done_v, _ = env.step_V(min(anchor + 1, 99))
assert obs3['turn'] == 'H'
assert 0.0 <= r_v <= 1.0

# Invalid H action
obs = env.reset(seed=SEED)
outside = next(f for f in range(100) if f not in obs['summary_indices'])
try:
    env.step_H(outside)
    assert False
except ValueError:
    pass

# Patience termination
obs = env.reset(seed=SEED)
done = False
while not done:
    if obs['turn'] == 'H':
        obs, _, done, _ = env.step_H(obs['summary_indices'][0])
    else:
        obs, _, done, _ = env.step_V(obs['anchor_idx'])  # always no-op
assert env.patience_counter >= cfg.stability_patience or env.step_count >= cfg.max_steps
print('Environment OK')

Environment OK


## 6. Policies

In [14]:
from horizontal_policy import HorizontalPolicy
from vertical_policy   import VerticalPolicy

h_pol = HorizontalPolicy(cfg.d_model, cfg.hidden_size).to(DEVICE)
v_pol = VerticalPolicy(cfg.d_model, cfg.hidden_size).to(DEVICE)

# H-policy
s_indices = [0, 10, 20, 30, 40]
s_feats   = torch.randn(len(s_indices), cfg.d_model).to(DEVICE)
anchor, lp_h = h_pol.select_anchor(s_feats, s_indices)
assert anchor in s_indices
assert lp_h.item() <= 0.0
print(f'H-policy → anchor={anchor}, log_prob={lp_h.item():.4f}')

# V-policy
ctx_np = np.random.randn(100, cfg.d_model).astype(np.float32)
for a in [0, 50, 99]:  # test boundary + midpoint
    nb, lp_v = v_pol.select_neighbor(a, ctx_np, cfg.window_size)
    assert 0 <= nb < 100
    assert lp_v.item() <= 0.0
    print(f'  V-policy anchor={a} → neighbor={nb}, log_prob={lp_v.item():.4f}')

print('Policies OK')

H-policy → anchor=40, log_prob=-1.5681
  V-policy anchor=0 → neighbor=5, log_prob=-1.7698
  V-policy anchor=50 → neighbor=49, log_prob=-2.4898
  V-policy anchor=99 → neighbor=98, log_prob=-1.8058
Policies OK


## 7. REINFORCE Trainer

In [15]:
from training.trainer import compute_returns, train_on_video

# compute_returns
rets = compute_returns([0.0, 1.0, 0.0, 0.5], gamma=1.0)
assert abs(rets[0].item() - 1.5) < 1e-5
assert abs(rets[2].item() - 0.5) < 1e-5
print('compute_returns:', rets.tolist(), '')

compute_returns: [1.5, 1.5, 0.5, 0.5] 


In [16]:
# RL-only episode
torch.manual_seed(SEED); np.random.seed(SEED)

ctx = np.random.randn(100, cfg.d_model).astype(np.float32)
env = VideoSummarizationEnv(ctx, cfg)
h, v = HorizontalPolicy(cfg.d_model, cfg.hidden_size), VerticalPolicy(cfg.d_model, cfg.hidden_size)
opt_h2 = torch.optim.Adam(h.parameters(), lr=cfg.lr)
opt_v2 = torch.optim.Adam(v.parameters(), lr=cfg.lr)

ep_r, summary = train_on_video(env, h, v, opt_h2, opt_v2, cfg)
print(f'Reward: {ep_r:.4f}  Summary size: {len(summary)}  Indices: {summary}')
assert len(summary) >= cfg.min_k
assert all(0 <= i < 100 for i in summary)
assert len(set(summary)) == len(summary), 'Duplicate indices!'
print('RL-only episode OK')

Reward: 5389.8477  Summary size: 3  Indices: [54, 65, 74]
RL-only episode OK


In [17]:
# End-to-end episode (transformer + policies)
torch.manual_seed(SEED); np.random.seed(SEED)

pipe2 = FeaturePipeline(cfg, device=DEVICE)
raw   = np.random.randn(100, cfg.transformer_input_dim).astype(np.float32)
ctx2  = pipe2.encode_video(raw)

env2  = VideoSummarizationEnv(ctx2, cfg)
h2, v2 = HorizontalPolicy(cfg.d_model, cfg.hidden_size), VerticalPolicy(cfg.d_model, cfg.hidden_size)
opt_h3 = torch.optim.Adam(h2.parameters(), lr=cfg.lr)
opt_v3 = torch.optim.Adam(v2.parameters(), lr=cfg.lr)
opt_t  = pipe2.get_optimizer(lr=cfg.lr)

ep_r2, summary2 = train_on_video(env2, h2, v2, opt_h3, opt_v3, cfg, pipeline=pipe2, opt_transformer=opt_t)
print(f'E2E Reward: {ep_r2:.4f}  Summary size: {len(summary2)}')
assert len(summary2) >= cfg.min_k
print('End-to-end episode OK')

/home/jovyan/RL-Video-Summarization/transformer/transformer_encoder.py:143: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


[FeaturePipeline] Transformer initialized on cuda
  Temporal Transformer Encoder
  Input dimension    : 768
  Output dimension   : 512
  Attention heads    : 8
  Encoder layers     : 3
  FFN hidden size    : 2048
  Total parameters   : 9,852,928
  Device             : cuda
E2E Reward: 5165.4541  Summary size: 3
End-to-end episode OK


## 8. Multi-video Training Loop (real data)

In [18]:
NUM_VIDEOS   = 3
NUM_EPISODES = 2
CATEGORY     = categories[0]

video_features = loader.load_category(CATEGORY, num_videos=NUM_VIDEOS, shuffle=True)
print(f'Loaded {len(video_features)} videos from "{CATEGORY}"')

[VideoEmbeddingLoader] Loaded 3 videos from 'Activity'
Loaded 3 videos from "Activity"


In [19]:
torch.manual_seed(SEED); np.random.seed(SEED)

pipe_m = FeaturePipeline(cfg, device=DEVICE)
h_m    = HorizontalPolicy(cfg.d_model, cfg.hidden_size)
v_m    = VerticalPolicy(cfg.d_model, cfg.hidden_size)
opt_hm = torch.optim.Adam(h_m.parameters(), lr=cfg.lr)
opt_vm = torch.optim.Adam(v_m.parameters(), lr=cfg.lr)

all_rewards = []

for ep in range(NUM_EPISODES):
    for vi, raw_f in enumerate(video_features):
        ctx_m = pipe_m.encode_video(raw_f)
        env_m = VideoSummarizationEnv(ctx_m, cfg)
        r, s  = train_on_video(env_m, h_m, v_m, opt_hm, opt_vm, cfg)
        all_rewards.append(r)
        print(f'  Ep {ep+1}/{NUM_EPISODES} | Video {vi+1}/{NUM_VIDEOS} → reward={r:.4f}, size={len(s)}')

print(f'\nMean reward: {np.mean(all_rewards):.4f}')
assert all(not np.isnan(r) for r in all_rewards), 'NaN reward!'
print('Multi-video loop OK')

[FeaturePipeline] Transformer initialized on cuda
  Temporal Transformer Encoder
  Input dimension    : 768
  Output dimension   : 512
  Attention heads    : 8
  Encoder layers     : 3
  FFN hidden size    : 2048
  Total parameters   : 9,852,928
  Device             : cuda
  Ep 1/2 | Video 1/3 → reward=5513.7739, size=4
  Ep 1/2 | Video 2/3 → reward=4916.4429, size=4
  Ep 1/2 | Video 3/3 → reward=5384.0933, size=6
  Ep 2/2 | Video 1/3 → reward=5469.0073, size=4
  Ep 2/2 | Video 2/3 → reward=5274.6724, size=4
  Ep 2/2 | Video 3/3 → reward=5329.1465, size=6

Mean reward: 5314.5227
Multi-video loop OK


# Full training

In [ ]:
!python train.py

[Trainer] Device: cuda
[Trainer] Checkpoint directory: checkpoints

RL VIDEO SUMMARIZATION — END-TO-END TRAINING
Config: Config(transformer_input_dim=768, transformer_d_model=512, transformer_nhead=8, transformer_num_layers=3, transformer_dim_feedfwd=2048, transformer_dropout=0.1, transformer_max_len=5000, d_model=512, hidden_size=256, alpha=0.02, min_k=3, max_k=20, window_size=5, max_steps=100, stability_patience=5, w_div=0.5, w_rep=0.5, lr=0.001, gamma=1.0, seed=42)

LOADING DATA
[VideoEmbeddingLoader] Found 3 categories
[VideoEmbeddingLoader] Categories: Activity, Animals, Incidents
[VideoEmbeddingLoader] Videos directory: /home/jovyan/RL-Video-Summarization/videos/embeddings_clip_vitl14

  Category: Activity
    Videos: 77
    Frame range: 22-466
    Avg frames: 253.4
[VideoEmbeddingLoader] Loaded 77 videos from 'Activity'

  Category: Animals
    Videos: 9
    Frame range: 25-158
    Avg frames: 88.0
[VideoEmbeddingLoader] Loaded 9 videos from 'Animals'

  Category: Incidents
    